In [11]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

ROOT = Path(r"C:\tcc\network-ids-ml-generalization")
MODEL_NAMES = ["decision_tree", "random_forest", "xgboost", "linear_svm", "mlp"]
models = {
    name: joblib.load(ROOT / "models" / "randomized_search" / name / "unsw_nb15.joblib")
    for name in MODEL_NAMES
}
features = list(models["linear_svm"].feature_names_in_)
test_df = pd.read_parquet(
    ROOT / "datasets" / "final" / "ml" / "cicids2017_test.parquet",
    columns=features + ["id", "label_binary"],
).reset_index(drop=True)
y_test = test_df["label_binary"].eq("ATTACK").astype(int)
predictions = pd.DataFrame(
    {name: model.predict(test_df[features]) for name, model in models.items()},
    index=test_df.index,
)


In [12]:
errors = []
for name in MODEL_NAMES:
    tn, fp, fn, tp = confusion_matrix(y_test, predictions[name], labels=[0, 1]).ravel()
    errors.append({
        "modelo": name,
        "f1_macro": f1_score(y_test, predictions[name], average="macro"),
        "falsos_positivos": fp,
        "falsos_negativos": fn,
    })
display(pd.DataFrame(errors).sort_values("f1_macro", ascending=False))

correct = predictions.eq(y_test, axis=0)
comparisons = []
for name in MODEL_NAMES:
    if name == "linear_svm":
        continue
    for label, class_name in [(0, "BENIGN"), (1, "ATTACK")]:
        mask = y_test.eq(label)
        comparisons.append({
            "concorrente": name,
            "classe_real": class_name,
            "svm_acerta_outro_erra": int((mask & correct["linear_svm"] & ~correct[name]).sum()),
            "svm_erra_outro_acerta": int((mask & ~correct["linear_svm"] & correct[name]).sum()),
        })
display(pd.DataFrame(comparisons))


,modelo,f1_macro,falsos_positivos,falsos_negativos
3,linear_svm,0.699412,10089,9528
4,mlp,0.631911,19636,8578
0,decision_tree,0.429723,24539,17084
1,random_forest,0.424111,28527,16279
2,xgboost,0.402903,22773,18649


,concorrente,classe_real,svm_acerta_outro_erra,svm_erra_outro_acerta
0,decision_tree,BENIGN,17541,3091
1,decision_tree,ATTACK,7721,165
2,random_forest,BENIGN,21008,2570
3,random_forest,ATTACK,7355,604
4,xgboost,BENIGN,15181,2497
5,xgboost,ATTACK,9458,337
6,mlp,BENIGN,11753,2206
7,mlp,ATTACK,89,1039


In [13]:
protocol_errors = []
for target in ["unsw_nb15", "cicids2017"]:
    target_df = test_df if target == "cicids2017" else pd.read_parquet(
        ROOT / "datasets" / "final" / "ml" / f"{target}_test.parquet",
        columns=features + ["label_binary"],
    ).reset_index(drop=True)
    target_y = target_df["label_binary"].eq("ATTACK").astype(int)
    target_predictions = predictions if target == "cicids2017" else pd.DataFrame(
        {name: model.predict(target_df[features]) for name, model in models.items()},
        index=target_df.index,
    )
    for protocol, group in target_df.groupby("protocol"):
        actual = target_y.loc[group.index]
        for name in MODEL_NAMES:
            tn, fp, fn, tp = confusion_matrix(
                actual, target_predictions.loc[group.index, name], labels=[0, 1]
            ).ravel()
            protocol_errors.append({
                "base_teste": target, "protocolo": protocol, "modelo": name,
                "benignos": tn + fp, "ataques": fn + tp,
                "falsos_positivos": fp, "falsos_negativos": fn,
                "taxa_falsos_positivos": fp / (tn + fp) if tn + fp else float("nan"),
                "recall_ataques": tp / (fn + tp) if fn + tp else float("nan"),
            })

protocol_errors_df = pd.DataFrame(protocol_errors)
display(protocol_errors_df.round(4))
svm_fn = (
    protocol_errors_df.query("base_teste == 'unsw_nb15' and modelo == 'linear_svm'")
    .set_index("protocolo")["falsos_negativos"]
)
print(f"Falsos negativos do SVM na UNSW que são UDP: {svm_fn.loc[17]} de {svm_fn.sum()} ({svm_fn.loc[17] / svm_fn.sum():.2%})")


,base_teste,protocolo,modelo,benignos,ataques,falsos_positivos,falsos_negativos,taxa_falsos_positivos,recall_ataques
0,unsw_nb15,6,decision_tree,57966,16081,943,67,0.0163,0.9958
1,unsw_nb15,6,random_forest,57966,16081,953,11,0.0164,0.9993
2,unsw_nb15,6,xgboost,57966,16081,946,20,0.0163,0.9988
3,unsw_nb15,6,linear_svm,57966,16081,1002,171,0.0173,0.9894
4,unsw_nb15,6,mlp,57966,16081,971,15,0.0168,0.9991
5,unsw_nb15,17,decision_tree,22102,4103,289,33,0.0131,0.9920
6,unsw_nb15,17,random_forest,22102,4103,292,19,0.0132,0.9954
7,unsw_nb15,17,xgboost,22102,4103,301,7,0.0136,0.9983
8,unsw_nb15,17,linear_svm,22102,4103,172,2661,0.0078,0.3515
9,unsw_nb15,17,mlp,22102,4103,323,38,0.0146,0.9907


Falsos negativos do SVM na UNSW que são UDP: 2661 de 2832 (93.96%)


In [14]:
train_udp = pd.read_parquet(
    ROOT / "datasets" / "final" / "ml" / "unsw_nb15_train.parquet",
    columns=features, filters=[("protocol", "=", 17), ("label_binary", "=", "BENIGN")],
)
udp = test_df.loc[test_df["protocol"].eq(17) & y_test.eq(0)]
groups = {
    "UNSW_UDP_benigno": train_udp,
    "CIC_UDP_benigno": udp,
    "CIC_SVM_acerta": udp.loc[predictions.loc[udp.index, "linear_svm"].eq(0)],
    "CIC_SVM_erra": udp.loc[predictions.loc[udp.index, "linear_svm"].eq(1)],
}
display(pd.Series({name: len(group) for name, group in groups.items()}, name="registros"))
numeric_features = [name for name in features if name != "protocol"]
display(pd.DataFrame({name: group[numeric_features].median() for name, group in groups.items()}))

svm = models["linear_svm"]
preprocessor = svm.named_steps["preprocessor"]
classifier = svm.named_steps["classifier"]
contributions = {}
for name, group in groups.items():
    transformed = preprocessor.transform(group[features])
    contributions[name] = (transformed * classifier.coef_[0]).mean(axis=0)
# Decomposição da margem no espaço transformado: f(x) = w.T @ z + b.
print("Intercepto do SVM:", classifier.intercept_[0])
display(pd.DataFrame(contributions, index=preprocessor.get_feature_names_out()).round(4))


UNSW_UDP_benigno    51916
CIC_UDP_benigno     29611
CIC_SVM_acerta      29221
CIC_SVM_erra          390
Name: registros, dtype: int64

,UNSW_UDP_benigno,CIC_UDP_benigno,CIC_SVM_acerta,CIC_SVM_erra
bidirectional_duration_ms,1.000000,31.000000,31.000000,1570.000000
bidirectional_packets,4.000000,4.000000,4.000000,4.000000
bidirectional_bytes,324.000000,342.000000,340.000000,876.000000
bidirectional_mean_ps,81.000000,100.000000,100.000000,137.875000
bidirectional_stddev_ps,9.237604,36.062446,36.373067,11.763248
bidirectional_max_ps,89.000000,131.000000,130.000000,192.000000
bidirectional_mean_piat_ms,0.333333,10.666667,10.666667,41.315789
bidirectional_stddev_piat_ms,0.577350,0.000000,0.000000,177.688217
bidirectional_max_piat_ms,1.000000,30.000000,30.000000,794.500000
bidirectional_syn_packets,0.000000,0.000000,0.000000,0.000000


Intercepto do SVM: -2.5772547864111535


,UNSW_UDP_benigno,CIC_UDP_benigno,CIC_SVM_acerta,CIC_SVM_erra
numerical__bidirectional_duration_ms,-0.0213,-0.0219,-0.0233,0.0824
numerical__bidirectional_packets,-2.9158,-2.9752,-2.9796,-2.6472
numerical__bidirectional_bytes,4.2214,4.2304,4.2325,4.0695
numerical__bidirectional_mean_ps,-0.3577,-0.3216,-0.3255,-0.0286
numerical__bidirectional_stddev_ps,1.2975,1.1700,1.1738,0.8833
numerical__bidirectional_max_ps,-1.1223,-1.0649,-1.0692,-0.7430
numerical__bidirectional_mean_piat_ms,-0.0002,0.0006,0.0004,0.0144
numerical__bidirectional_stddev_piat_ms,-0.0160,-0.0022,-0.0102,0.6021
numerical__bidirectional_max_piat_ms,0.0403,0.0252,0.0314,-0.4362
numerical__bidirectional_syn_packets,0.2469,0.2469,0.2469,0.2469


In [15]:
shap_root = ROOT / "results" / "shap_analysis"
svm_profiles = {
    target: pd.read_csv(
        shap_root / "linear_svm" / f"unsw_nb15__{target}" / "global_importance.csv"
    ).set_index("feature")[["mean_absolute_shap", "mean_signed_shap"]]
    for target in ["unsw_nb15", "cicids2017"]
}
print("SVM treinado em UNSW-NB15: SHAP intra-dataset e cross-dataset, na escala da margem.")
display(pd.concat(svm_profiles, axis=1).round(4))

svm_values = pd.read_csv(
    shap_root / "linear_svm" / "unsw_nb15__cicids2017" / "shap_values.csv"
).pivot(index="row_id", columns="feature", values="shap_value")
for name in MODEL_NAMES:
    if name == "linear_svm":
        continue
    other_values = pd.read_csv(
        shap_root / name / "unsw_nb15__cicids2017" / "shap_values.csv"
    ).pivot(index="row_id", columns="feature", values="shap_value")
    pair_groups = {
        "SVM_acerta_outro_erra": udp.index[
            correct.loc[udp.index, "linear_svm"] & ~correct.loc[udp.index, name]
        ],
        "ambos_acertam": udp.index[correct.loc[udp.index, ["linear_svm", name]].all(axis=1)],
    }
    pair_means, counts = {}, {}
    for group, indices in pair_groups.items():
        rows = svm_values.index.intersection(other_values.index).intersection(indices)
        counts[group] = len(rows)
        pair_means[("SVM_margem", group)] = svm_values.loc[rows].mean()
        pair_means[(f"{name}_probabilidade", group)] = other_values.loc[rows].mean()
    print(name, "— UDP benigno explicado:", counts)
    # SVM: margem de decisão; demais modelos: probabilidade estimada de ATTACK.
    display(pd.DataFrame(pair_means).round(4))


SVM treinado em UNSW-NB15: SHAP intra-dataset e cross-dataset, na escala da margem.


unsw_nb15                   \
                             mean_absolute_shap mean_signed_shap   
feature                                                            
bidirectional_psh_packets                8.8829          -1.3759   
bidirectional_ack_packets                6.3393           1.2842   
bidirectional_bytes                      4.5177          -1.2643   
bidirectional_packets                    2.7391           0.5640   
bidirectional_stddev_ps                  1.1768          -0.0248   
bidirectional_max_ps                     0.9099           0.0128   
bidirectional_mean_ps                    0.3849           0.0367   
bidirectional_syn_packets                0.1367          -0.0066   
protocol                                 0.0621           0.0004   
bidirectional_fin_packets                0.0600           0.0039   
bidirectional_max_piat_ms                0.0535          -0.0117   
bidirectional_stddev_piat_ms             0.0255           0.0051   
bidirectional_duration_ms                0.0232           0.0085   
bidirectional_mean_piat_ms               0.0004           0.0001   
bidirectional_rst_packets                0.0000          -0.0000   

                                     cicids2017                   
                             mean_absolute_shap mean_signed_shap  
feature                                                           
bidirectional_psh_packets                6.4466           6.3631  
bidirectional_ack_packets                4.7735          -4.5277  
bidirectional_bytes                      2.5739           2.2111  
bidirectional_packets                    2.1172          -2.0119  
bidirectional_stddev_ps                  1.9899          -0.0545  
bidirectional_max_ps                     2.0128           0.3019  
bidirectional_mean_ps                    0.4859          -0.0678  
bidirectional_syn_packets                0.1878           0.1860  
protocol                                 0.0609           0.0135  
bidirectional_fin_packets                0.0857          -0.0854  
bidirectional_max_piat_ms                0.2275          -0.1884  
bidirectional_stddev_piat_ms             0.2068           0.1863  
bidirectional_duration_ms                0.0664           0.0498  
bidirectional_mean_piat_ms               0.0041           0.0040  
bidirectional_rst_packets                0.0471           0.0471

decision_tree — UDP benigno explicado: {'SVM_acerta_outro_erra': 121, 'ambos_acertam': 180}


,SVM_margem,decision_tree_probabilidade,SVM_margem,decision_tree_probabilidade
,SVM_acerta_outro_erra,SVM_acerta_outro_erra,ambos_acertam,ambos_acertam
feature,,,,
bidirectional_ack_packets,-5.1341,-0.0075,-5.1341,-0.0096
bidirectional_bytes,2.6101,0.0404,2.6051,0.0283
bidirectional_duration_ms,-0.0080,0.0261,-0.0113,-0.0046
bidirectional_fin_packets,-0.1028,-0.0077,-0.1028,-0.0036
bidirectional_max_piat_ms,0.0089,0.0353,0.0261,0.0022
bidirectional_max_ps,-1.0221,0.0179,-1.0477,-0.0191
bidirectional_mean_piat_ms,0.0012,0.3406,0.0001,-0.0634
bidirectional_mean_ps,-0.2724,0.0448,-0.2936,-0.0363


random_forest — UDP benigno explicado: {'SVM_acerta_outro_erra': 139, 'ambos_acertam': 162}


,SVM_margem,random_forest_probabilidade,SVM_margem,random_forest_probabilidade
,SVM_acerta_outro_erra,SVM_acerta_outro_erra,ambos_acertam,ambos_acertam
feature,,,,
bidirectional_ack_packets,-5.1341,-0.0197,-5.1341,-0.0119
bidirectional_bytes,2.6127,0.0646,2.6024,0.0236
bidirectional_duration_ms,-0.0075,-0.0088,-0.0121,-0.0216
bidirectional_fin_packets,-0.1028,-0.0207,-0.1028,-0.0128
bidirectional_max_piat_ms,0.0058,0.0360,0.0307,-0.0151
bidirectional_max_ps,-1.0385,0.0150,-1.0365,-0.0018
bidirectional_mean_piat_ms,0.0013,0.1334,-0.0001,-0.0158
bidirectional_mean_ps,-0.2833,0.0125,-0.2866,-0.0015


xgboost — UDP benigno explicado: {'SVM_acerta_outro_erra': 100, 'ambos_acertam': 201}


,SVM_margem,xgboost_probabilidade,SVM_margem,xgboost_probabilidade
,SVM_acerta_outro_erra,SVM_acerta_outro_erra,ambos_acertam,ambos_acertam
feature,,,,
bidirectional_ack_packets,-5.1341,0.0034,-5.1341,-0.0068
bidirectional_bytes,2.6097,0.0314,2.6059,-0.0016
bidirectional_duration_ms,-0.0080,-0.0110,-0.0110,-0.0231
bidirectional_fin_packets,-0.1028,-0.0010,-0.1028,-0.0009
bidirectional_max_piat_ms,0.0066,0.0247,0.0255,-0.0019
bidirectional_max_ps,-1.0161,0.0145,-1.0480,-0.0100
bidirectional_mean_piat_ms,0.0012,0.2162,0.0002,-0.0317
bidirectional_mean_ps,-0.2692,-0.0013,-0.2930,-0.0293


mlp — UDP benigno explicado: {'SVM_acerta_outro_erra': 54, 'ambos_acertam': 247}


,SVM_margem,mlp_probabilidade,SVM_margem,mlp_probabilidade
,SVM_acerta_outro_erra,SVM_acerta_outro_erra,ambos_acertam,ambos_acertam
feature,,,,
bidirectional_ack_packets,-5.1341,-0.0867,-5.1341,-0.1081
bidirectional_bytes,2.6180,-0.0091,2.6048,-0.0093
bidirectional_duration_ms,-0.0061,-0.0203,-0.0108,-0.0406
bidirectional_fin_packets,-0.1028,-0.0872,-0.1028,-0.1561
bidirectional_max_piat_ms,-0.0048,-0.0072,0.0244,-0.0123
bidirectional_max_ps,-1.0442,-0.1335,-1.0359,-0.1808
bidirectional_mean_piat_ms,0.0019,0.2553,0.0002,0.0180
bidirectional_mean_ps,-0.2844,-0.0342,-0.2853,-0.0405
